# 00 — Data Quality

This notebook is the **data-quality gate** for the Moneyball season-wins project.

### Objectives

1. Validate the raw train and prediction files.
2. Check schema, keys, missing values, and basic statistical ranges.
3. Diagnose whether `teamID`, `franchID`, and `yearID` can be trusted as identity fields.
4. Compare rows with the Lahman `Teams` reference when it is available.
5. Decide which fields are safe for downstream EDA and modelling.

> **Important:** reconciliation is diagnostic only. This notebook does not silently rewrite the raw data. Identity fields must not be used for lag/franchise-history features unless they are first reconciled unambiguously.

In [2]:
import json

import numpy as np
import pandas as pd
from pathlib import Path

from moneyball import project_config as cfg

# ---------------------------------------------------------------------
# Shared project configuration
# ---------------------------------------------------------------------

cfg.configure_notebook()
# cfg.ensure_project_dirs()

# Data-quality review needs to display more rows than the normal notebooks.
pd.set_option("display.max_rows", 100)

# ---------------------------------------------------------------------
# Local aliases
# --------------------------------------------------------------------_

PROJECT_ROOT  = cfg.PROJECT_ROOT
RAW_DIR       = cfg.RAW_DIR

TRAIN_PATH    = cfg.TRAIN_PATH
PRED_PATH     = cfg.PRED_PATH

ID_COL        = cfg.ID_COL
YEAR_COL      = cfg.YEAR_COL
TEAM_COL      = cfg.TEAM_COL
FRANCHISE_COL = cfg.FRANCHISE_COL
TARGET_COL    = cfg.TARGET_COL
GAMES_COL     = cfg.GAMES_COL


# ---------------------------------------------------------------------
# Data-quality output
# ---------------------------------------------------------------------
REPORT_DIR = cfg.OUTPUT_DIR / "data_quality"
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Data-quality rules
# ---------------------------------------------------------------------
ALLOWED_TRAIN_ONLY_COLUMNS = {
    TARGET_COL,
    "year_label",
    "decade_label",
    "win_bins",
}

COUNTING_STATS = [
    "G", "R", "AB", "H", "2B", "3B", "HR", "BB", "SO", "SB",
    "RA", "ER", "CG", "SHO", "SV", "IPouts", "HA", "HRA",
    "BBA", "SOA", "E", "DP",
]

PREFERRED_MATCH_COLS = [
    "G", "R", "AB", "H", "2B", "3B", "HR", "BB", "SO", "SB",
    "RA", "ER", "ERA", "CG", "SHO", "SV", "IPouts",
    "HA", "HRA", "BBA", "SOA", "E", "DP", "FP",
]

FLOAT_ROUNDING = {
    "ERA": 3,
    "FP": 3,
}


# ---------------------------------------------------------------------
# Traceability
# ---------------------------------------------------------------------

cfg.show_project_paths()
print(f"Report dir   : {REPORT_DIR}")

Project root : /home/shpang/devs/ntu/projects/baseball_v2
Train        : /home/shpang/devs/ntu/projects/baseball_v2/data/raw/data_year_team_franchise.csv
Prediction   : /home/shpang/devs/ntu/projects/baseball_v2/data/raw/predict_year_team_franchise.csv
Processed    : /home/shpang/devs/ntu/projects/baseball_v2/data/processed
Report dir   : /home/shpang/devs/ntu/projects/baseball_v2/outputs/data_quality


In [9]:
# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")


def duplicate_key_count(df: pd.DataFrame, keys: list[str]) -> int:
    return int(
        df.loc[df.duplicated(keys, keep=False), keys]
        .drop_duplicates()
        .shape[0]
    )


def cross_file_overlap(
    train_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    keys: list[str],
) -> pd.DataFrame:
    return (
        train_df[keys].drop_duplicates()
        .merge(pred_df[keys].drop_duplicates(), on=keys, how="inner")
        .sort_values(keys)
        .reset_index(drop=True)
    )


def compare_overlapping_rows(
    train_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    keys: list[str],
    ignored_columns: set[str] | None = None,
) -> pd.DataFrame:
    ignored_columns = ignored_columns or set()
    common = [
        c for c in train_df.columns
        if c in pred_df.columns and c not in set(keys) | ignored_columns
    ]

    merged = train_df.merge(
        pred_df,
        on=keys,
        how="inner",
        suffixes=("_train", "_pred"),
    )
    if merged.empty:
        return merged

    flags = {}
    for col in common:
        left = merged[f"{col}_train"]
        right = merged[f"{col}_pred"]

        if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
            equal = np.isclose(
                pd.to_numeric(left, errors="coerce"),
                pd.to_numeric(right, errors="coerce"),
                equal_nan=True,
            )
        else:
            equal = left.fillna("<NA>").astype(str).eq(
                right.fillna("<NA>").astype(str)
            ).to_numpy()

        flags[col] = ~equal

    diff = pd.DataFrame(flags, index=merged.index)
    merged["different_column_count"] = diff.sum(axis=1).astype(int)
    merged["different_columns"] = diff.apply(
        lambda row: ", ".join(row.index[row].tolist()),
        axis=1,
    )
    merged["rows_identical_on_common_data"] = (
        merged["different_column_count"].eq(0)
    )

    front = keys + [
        "different_column_count",
        "different_columns",
        "rows_identical_on_common_data",
    ]
    return merged[front + [c for c in merged.columns if c not in front]]


def normalise_component(
    series: pd.Series,
    column: str,
    float_rounding: dict[str, int],
) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")

    if column in float_rounding:
        digits = float_rounding[column]
        return numeric.round(digits).map(
            lambda value: "<NA>" if pd.isna(value) else f"{value:.{digits}f}"
        )

    rounded = numeric.round()
    non_integer = numeric.notna() & ~np.isclose(numeric, rounded, atol=1e-9)

    output = pd.Series(index=series.index, dtype="object")
    output.loc[numeric.isna()] = "<NA>"
    output.loc[numeric.notna() & ~non_integer] = (
        rounded.loc[numeric.notna() & ~non_integer]
        .astype("Int64")
        .astype(str)
    )
    output.loc[non_integer] = numeric.loc[non_integer].map(
        lambda value: f"{value:.8g}"
    )
    return output


def build_signature(
    df: pd.DataFrame,
    columns: list[str],
    float_rounding: dict[str, int],
) -> pd.Series:
    parts = [
        normalise_component(df[column], column, float_rounding)
        .astype(str)
        .rename(column)
        for column in columns
    ]
    return pd.concat(parts, axis=1).agg("|".join, axis=1)


def identity_action(row: pd.Series) -> str:
    changed = []

    raw_year = pd.to_numeric(pd.Series([row["raw_yearID"]]), errors="coerce").iloc[0]
    ref_year = pd.to_numeric(pd.Series([row["ref_yearID"]]), errors="coerce").iloc[0]
    if not (pd.isna(raw_year) and pd.isna(ref_year)) and raw_year != ref_year:
        changed.append("yearID")

    for raw_col, ref_col, label in [
        ("raw_teamID", "ref_teamID", "teamID"),
        ("raw_franchID", "ref_franchID", "franchID"),
    ]:
        if str(row[raw_col]).strip().upper() != str(row[ref_col]).strip().upper():
            changed.append(label)

    if not changed:
        return "already_correct"
    if len(changed) == 1:
        return f"{changed[0]}_differs"
    return "multiple_identity_fields_differ"


def add_issue(
    issues: list[dict],
    severity: str,
    check: str,
    count: int,
    message: str,
) -> None:
    issues.append({
        "severity": severity,
        "check": check,
        "count": int(count),
        "message": message,
    })


## 1. Load and profile the raw datasets

**NOTE:** The training and prediction files are immutable raw inputs.

In [10]:
require_file(TRAIN_PATH)
require_file(PRED_PATH)

train = pd.read_csv(TRAIN_PATH)
pred = pd.read_csv(PRED_PATH)

train["_source"] = "train"
pred["_source"] = "pred"

combined = pd.concat([train, pred], ignore_index=True, sort=False)

dataset_summary = pd.DataFrame([
    {
        "dataset": "train",
        "rows": len(train),
        "columns": train.shape[1] - 1,
        "year_min": pd.to_numeric(train.get(YEAR_COL), errors="coerce").min(),
        "year_max": pd.to_numeric(train.get(YEAR_COL), errors="coerce").max(),
    },
    {
        "dataset": "prediction",
        "rows": len(pred),
        "columns": pred.shape[1] - 1,
        "year_min": pd.to_numeric(pred.get(YEAR_COL), errors="coerce").min(),
        "year_max": pd.to_numeric(pred.get(YEAR_COL), errors="coerce").max(),
    },
])

display(dataset_summary)
display(train.drop(columns="_source").head(3))


,dataset,rows,columns,year_min,year_max
0,train,1812,52,1904,2016
1,prediction,453,48,1892,2016


,yearID,teamID,G,R,AB,H,2B,3B,HR,BB,SO,SB,RA,ER,ERA,CG,SHO,SV,IPouts,HA,HRA,BBA,SOA,E,DP,FP,mlb_rpg,era_1,era_2,era_3,era_4,era_5,era_6,era_7,era_8,decade_1910,decade_1920,decade_1930,decade_1940,decade_1950,decade_1960,decade_1970,decade_1980,decade_1990,decade_2000,decade_2010,W,ID,year_label,decade_label,win_bins,franchID
0,1935,BOS,154,718,5288,1458,281,63,69,609,470.0,91,732,619,4.05,82,6,11,4128,1520,67,520,470,190,136.0,0.969,4.864690,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,78,317,1935,1930s,3,BOS
1,1993,TEX,162,835,5510,1472,284,39,181,483,984.0,113,751,684,4.28,20,6,45,4314,1476,144,562,957,130,145.0,0.979,4.597620,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,86,2162,1993,1990s,3,TEX
2,2016,SEA,162,768,5583,1446,251,17,223,506,1288.0,56,707,647,4.00,2,8,49,4371,1410,213,460,1318,89,158.0,0.985,4.477759,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,86,1895,2016,2010s,3,SEA


In [11]:
# ---------------------------------------------------------------------
# Load raw files
# ---------------------------------------------------------------------

require_file(TRAIN_PATH)
require_file(PRED_PATH)

train = pd.read_csv(TRAIN_PATH)
pred = pd.read_csv(PRED_PATH)

train["_source"] = "train"
pred["_source"] = "pred"

print(f"Train shape   : {train.shape}")
print(f"Predict shape : {pred.shape}")

display(train.head(3))
display(pred.head(3))

Train shape   : (1812, 53)
Predict shape : (453, 49)


,yearID,teamID,G,R,AB,H,2B,3B,HR,BB,SO,SB,RA,ER,ERA,CG,SHO,SV,IPouts,HA,HRA,BBA,SOA,E,DP,FP,mlb_rpg,era_1,era_2,era_3,era_4,era_5,era_6,era_7,era_8,decade_1910,decade_1920,decade_1930,decade_1940,decade_1950,decade_1960,decade_1970,decade_1980,decade_1990,decade_2000,decade_2010,W,ID,year_label,decade_label,win_bins,franchID,_source
0,1935,BOS,154,718,5288,1458,281,63,69,609,470.0,91,732,619,4.05,82,6,11,4128,1520,67,520,470,190,136.0,0.969,4.864690,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,78,317,1935,1930s,3,BOS,train
1,1993,TEX,162,835,5510,1472,284,39,181,483,984.0,113,751,684,4.28,20,6,45,4314,1476,144,562,957,130,145.0,0.979,4.597620,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,86,2162,1993,1990s,3,TEX,train
2,2016,SEA,162,768,5583,1446,251,17,223,506,1288.0,56,707,647,4.00,2,8,49,4371,1410,213,460,1318,89,158.0,0.985,4.477759,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,86,1895,2016,2010s,3,SEA,train


,G,R,AB,H,2B,3B,HR,BB,SO,SB,RA,ER,ERA,CG,SHO,SV,IPouts,HA,HRA,BBA,SOA,E,DP,FP,mlb_rpg,era_1,era_2,era_3,era_4,era_5,era_6,era_7,era_8,decade_1910,decade_1920,decade_1930,decade_1940,decade_1950,decade_1960,decade_1970,decade_1980,decade_1990,decade_2000,decade_2010,ID,yearID,teamID,franchID,_source
0,157,588,5221,1340,199,57,110,383,752.0,24,653,572,3.74,37,8,24,4128,1406,142,469,662,162,140.0,0.973,4.451574,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,1756,1940,PIT,PIT,pred
1,161,707,5417,1353,215,40,167,597,840.0,47,778,681,4.28,49,14,23,4296,1415,163,570,914,174,150.0,0.971,4.525175,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,1282,1951,WS1,MIN,pred
2,162,743,5494,1381,234,37,197,658,923.0,41,736,639,3.92,30,7,41,4398,1423,155,685,935,157,178.0,0.975,4.072456,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,351,1966,BOS,BOS,pred


## 2. Schema, missing values, and basic sanity checks

Checks to determine whether the numeric season data is structurally usable for EDA and modelling. Identity anomalies are assessed separately in the next section.


In [7]:
issues = []

required_common = {
    ID_COL,
    YEAR_COL,
    TEAM_COL,
    FRANCHISE_COL,
    GAMES_COL,
}

missing_train_required = sorted(required_common - set(train.columns))
missing_pred_required = sorted(required_common - set(pred.columns))

if missing_train_required:
    add_issue(
        issues, "BLOCKER", "required_columns_train",
        len(missing_train_required),
        f"Missing required train columns: {missing_train_required}",
    )

if missing_pred_required:
    add_issue(
        issues, "BLOCKER", "required_columns_pred",
        len(missing_pred_required),
        f"Missing required prediction columns: {missing_pred_required}",
    )

if TARGET_COL not in train.columns:
    add_issue(
        issues, "BLOCKER", "target_missing_in_train", 1,
        f"Training file does not contain target column {TARGET_COL!r}.",
    )

if TARGET_COL in pred.columns and pred[TARGET_COL].notna().any():
    add_issue(
        issues, "BLOCKER", "target_present_in_pred",
        int(pred[TARGET_COL].notna().sum()),
        f"Prediction file must not expose target {TARGET_COL!r}.",
    )

train_only = sorted((set(train.columns) - set(pred.columns)) - {"_source"})
pred_only = sorted((set(pred.columns) - set(train.columns)) - {"_source"})

unexpected_train_only = sorted(set(train_only) - ALLOWED_TRAIN_ONLY_COLUMNS)
if unexpected_train_only:
    add_issue(
        issues, "WARNING", "unexpected_train_only_columns",
        len(unexpected_train_only),
        f"Unexpected train-only columns: {unexpected_train_only}",
    )

if pred_only:
    add_issue(
        issues, "WARNING", "prediction_only_columns",
        len(pred_only),
        f"Prediction-only columns: {pred_only}",
    )

missingness = pd.concat(
    {
        "train": train.drop(columns="_source").isna().sum(),
        "prediction": pred.drop(columns="_source").isna().sum(),
    },
    axis=1,
).fillna(0).astype(int)

missingness["train_pct"] = (missingness["train"] / max(len(train), 1) * 100).round(2)
missingness["prediction_pct"] = (
    missingness["prediction"] / max(len(pred), 1) * 100
).round(2)
missingness = missingness.reset_index(names="column")

for source_name, df in [("train", train), ("prediction", pred)]:
    for col in required_common:
        if col in df.columns and df[col].isna().any():
            add_issue(
                issues, "BLOCKER", f"missing_{source_name}_{col}",
                int(df[col].isna().sum()),
                f"{source_name.title()} has missing values in required field {col}.",
            )

    if GAMES_COL in df.columns:
        games = pd.to_numeric(df[GAMES_COL], errors="coerce")
        bad = games.isna() | (games <= 0)
        if bad.any():
            add_issue(
                issues, "BLOCKER", f"{source_name}_invalid_games",
                int(bad.sum()), "Games played must be positive.",
            )

    if YEAR_COL in df.columns:
        years = pd.to_numeric(df[YEAR_COL], errors="coerce")
        bad = years.isna() | (years < 1800) | (years > 2100)
        if bad.any():
            add_issue(
                issues, "BLOCKER", f"{source_name}_invalid_year",
                int(bad.sum()), "yearID contains invalid values.",
            )

    present_counts = [c for c in COUNTING_STATS if c in df.columns]
    if present_counts:
        numeric_counts = df[present_counts].apply(pd.to_numeric, errors="coerce")
        bad = numeric_counts.lt(0).any(axis=1)
        if bad.any():
            add_issue(
                issues, "BLOCKER", f"{source_name}_negative_counting_stats",
                int(bad.sum()), "Counting statistics must not be negative.",
            )

if TARGET_COL in train.columns and GAMES_COL in train.columns:
    wins = pd.to_numeric(train[TARGET_COL], errors="coerce")
    games = pd.to_numeric(train[GAMES_COL], errors="coerce")
    bad = wins.isna() | (wins < 0) | (wins > games)
    if bad.any():
        add_issue(
            issues, "BLOCKER", "wins_outside_0_to_G",
            int(bad.sum()), "Wins must be between 0 and games played.",
        )

schema_summary = pd.DataFrame(
    {
        "metric": [
            "train_rows",
            "train_columns",
            "prediction_rows",
            "prediction_columns",
            "train_only_columns",
            "prediction_only_columns",
        ],
        "value": [
            len(train),
            train.shape[1] - 1,
            len(pred),
            pred.shape[1] - 1,
            ", ".join(train_only) or "<none>",
            ", ".join(pred_only) or "<none>",
        ],
    }
)

display(schema_summary)

missing_display = missingness.loc[
    (missingness["train"] > 0) | (missingness["prediction"] > 0)
]
if missing_display.empty:
    print("No missing values found.")
else:
    display(missing_display)

,metric,value
0,train_rows,1812
1,train_columns,52
2,prediction_rows,453
3,prediction_columns,48
4,train_only_columns,"W, decade_label, win_bins, year_label"
5,prediction_only_columns,<none>


No missing values found.


## 3. Identity and split-integrity diagnostics

`ID` should be unique. `teamID + yearID` and `franchID + yearID` are
examined separately because they are baseball identity fields rather than model measurements.

If these identity keys conflict while the numeric season statistics remain valid, the project can still proceed **provided the unreliable identity fields are not used as predictors or to build lag/history features**.


In [9]:
# IDs are structural keys and therefore remain hard validation checks.
train_id_dups = duplicate_key_count(train, [ID_COL])
pred_id_dups = duplicate_key_count(pred, [ID_COL])
id_overlap = cross_file_overlap(train, pred, [ID_COL])

if train_id_dups:
    add_issue(
        issues, "BLOCKER", "duplicate_train_ids", train_id_dups,
        "Training contains duplicated IDs.",
    )
if pred_id_dups:
    add_issue(
        issues, "BLOCKER", "duplicate_prediction_ids", pred_id_dups,
        "Prediction contains duplicated IDs.",
    )
if len(id_overlap):
    add_issue(
        issues, "BLOCKER", "cross_file_id_overlap", len(id_overlap),
        "The same ID appears in both train and prediction.",
    )

# Baseball identity keys are diagnostic constraints rather than automatic blockers.
train_team_year_dups = duplicate_key_count(train, [TEAM_COL, YEAR_COL])
pred_team_year_dups = duplicate_key_count(pred, [TEAM_COL, YEAR_COL])
cross_team_year = cross_file_overlap(train, pred, [TEAM_COL, YEAR_COL])

train_franchise_year_dups = duplicate_key_count(
    train, [FRANCHISE_COL, YEAR_COL]
)
pred_franchise_year_dups = duplicate_key_count(
    pred, [FRANCHISE_COL, YEAR_COL]
)
cross_franchise_year = cross_file_overlap(
    train, pred, [FRANCHISE_COL, YEAR_COL]
)

combined_franchise_year_dups = duplicate_key_count(
    combined, [FRANCHISE_COL, YEAR_COL]
)

if len(cross_team_year):
    add_issue(
        issues, "WARNING", "cross_file_team_year_overlap",
        len(cross_team_year),
        "teamID + yearID overlaps across train/prediction; identity metadata "
        "must not be assumed reliable.",
    )

if combined_franchise_year_dups:
    add_issue(
        issues, "WARNING", "ambiguous_franchise_year",
        combined_franchise_year_dups,
        "franchID + yearID is not unique; franchise-based lag features are unsafe.",
    )

key_summary = pd.DataFrame([
    ["Duplicate train IDs", train_id_dups],
    ["Duplicate prediction IDs", pred_id_dups],
    ["IDs in both files", len(id_overlap)],
    ["Duplicate train team-years", train_team_year_dups],
    ["Duplicate prediction team-years", pred_team_year_dups],
    ["Team-years in both files", len(cross_team_year)],
    ["Duplicate train franchise-years", train_franchise_year_dups],
    ["Duplicate prediction franchise-years", pred_franchise_year_dups],
    ["Franchise-years in both files", len(cross_franchise_year)],
    ["Duplicate combined franchise-years", combined_franchise_year_dups],
], columns=["check", "count"])

display(key_summary)

overlap_comparison = compare_overlapping_rows(
    train.drop(columns="_source"),
    pred.drop(columns="_source"),
    keys=[TEAM_COL, YEAR_COL],
    ignored_columns={ID_COL, FRANCHISE_COL, TARGET_COL},
)

if overlap_comparison.empty:
    print("No train/prediction team-year overlaps found.")
else:
    overlap_summary = pd.DataFrame({
        "metric": [
            "overlapping_joined_rows",
            "identical_on_common_data",
            "conflicting_on_common_data",
        ],
        "count": [
            len(overlap_comparison),
            int(overlap_comparison["rows_identical_on_common_data"].sum()),
            int((~overlap_comparison["rows_identical_on_common_data"]).sum()),
        ],
    })
    display(overlap_summary)
    display(
        overlap_comparison[
            [
                TEAM_COL,
                YEAR_COL,
                "different_column_count",
                "different_columns",
                "rows_identical_on_common_data",
            ]
        ].head(15)
    )


,check,count
0,Duplicate train IDs,0
1,Duplicate prediction IDs,0
2,IDs in both files,0
3,Duplicate train team-years,0
4,Duplicate prediction team-years,0
5,Team-years in both files,361
6,Duplicate train franchise-years,78
7,Duplicate prediction franchise-years,0
8,Franchise-years in both files,320
9,Duplicate combined franchise-years,384


,metric,count
0,overlapping_joined_rows,361
1,identical_on_common_data,0
2,conflicting_on_common_data,361


,teamID,yearID,different_column_count,different_columns,rows_identical_on_common_data
0,CHN,1938,27,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False
1,ML1,1955,24,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False
2,PHA,1938,28,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False
3,SDN,1990,28,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False
4,NY1,1920,27,"G, R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, E...",False
5,DET,1990,28,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False
6,SFN,1959,28,"G, R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, E...",False
7,CIN,1916,28,"G, R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, E...",False
8,FLO,2006,28,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False
9,CIN,2007,28,"R, AB, H, 2B, 3B, HR, BB, SO, SB, RA, ER, ERA,...",False


## 4. Chronology diagnostics for lag engineering

This does not create lag features. It checks whether an exact prior-year record exists after train and prediction are combined.

A proper lag should match:

```text
(entity, current_year - 1)
```

It should not silently use the previous available row from two or more years earlier.

In [10]:
chronology_base = combined[
    [FRANCHISE_COL, YEAR_COL, ID_COL, "_source"]
].dropna(subset=[FRANCHISE_COL, YEAR_COL]).copy()

chronology_base[YEAR_COL] = pd.to_numeric(
    chronology_base[YEAR_COL], errors="coerce"
).astype("Int64")

unique_entity_year = (
    chronology_base[[FRANCHISE_COL, YEAR_COL]]
    .drop_duplicates()
    .dropna()
)

prior_keys = unique_entity_year.copy()
prior_keys[YEAR_COL] = prior_keys[YEAR_COL] + 1
prior_keys["exact_prior_year_available"] = True

chronology_check = chronology_base.merge(
    prior_keys,
    on=[FRANCHISE_COL, YEAR_COL],
    how="left",
)

chronology_check["exact_prior_year_available"] = (
    chronology_check["exact_prior_year_available"]
    .eq(True)
)

chronology_summary = (
    chronology_check
    .groupby("_source")["exact_prior_year_available"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "rows",
        "sum": "rows_with_exact_prior_year",
        "mean": "exact_prior_year_rate",
    })
    .reset_index()
)
chronology_summary["exact_prior_year_rate"] = (
    chronology_summary["exact_prior_year_rate"] * 100
).round(2)

display(chronology_summary)

if combined_franchise_year_dups:
    print(
        "Lag-feature decision: DO NOT use franchise-based lag features. "
        "franchID + yearID is ambiguous in the combined data."
    )
else:
    print(
        "Lag-feature key uniqueness check passed. Historical features would still "
        "need leakage-safe construction inside each validation fold."
    )


,_source,rows,rows_with_exact_prior_year,exact_prior_year_rate
0,pred,453,339,74.83
1,train,1812,1467,80.96


Lag-feature decision: DO NOT use franchise-based lag features. franchID + yearID is ambiguous in the combined data.


## 5. Final data-quality decision

The final decision separates **structural blockers** from **identity constraints**.

A project can proceed to EDA/model selection when the numeric season data and target are structurally valid, even if team/franchise identity metadata is unreliable provided those identity fields are excluded from the model and are not used for historical lag features.


In [12]:
issues_df = pd.DataFrame(
    issues,
    columns=["severity", "check", "count", "message"],
)

if not issues_df.empty:
    severity_rank = {"BLOCKER": 0, "WARNING": 1, "INFO": 2}
    issues_df["_rank"] = issues_df["severity"].map(severity_rank)
    issues_df = (
        issues_df.sort_values(["_rank", "check"])
        .drop(columns="_rank")
        .reset_index(drop=True)
    )

blocker_count = (
    int(issues_df["severity"].eq("BLOCKER").sum())
    if not issues_df.empty else 0
)
warning_count = (
    int(issues_df["severity"].eq("WARNING").sum())
    if not issues_df.empty else 0
)

identity_constrained = bool(
    len(cross_team_year)
    or combined_franchise_year_dups
)

if blocker_count:
    status = "STOP — STRUCTURAL BLOCKERS"
elif warning_count:
    status = "USABLE WITH CAVEATS"
elif identity_constrained or warning_count:
    status = "PROCEED WITH CONSTRAINTS"
else:
    status = "PASS"

decision = pd.DataFrame([
    ["Data-quality status", status],
    ["Structural blocker checks", blocker_count],
    ["Warning checks", warning_count],
    ["Use ID as row identifier", True],
    ["Use numeric season statistics for EDA/modeling", blocker_count == 0],
    ["Use W as supervised target", blocker_count == 0 and TARGET_COL in train.columns],
    ["Use teamID/franchID as model predictors", False],
    ["Use yearID for EDA and year-aware validation", True],
    ["Use franchise/team lag features", False],
], columns=["decision", "value"])

display(decision)

if not issues_df.empty:
    display(issues_df)

print("\nInterpretation:")
print("1. ID is the unique row-level identifier.")
print("2. teamID, franchID and yearID are descriptive fields, not global row keys.")
print("3. teamID + yearID overlaps across train and prediction, so that pair is not a global observation key.")
print("4. Some franchID + yearID combinations are not unique.")
print("5. Lag features are excluded because historical linkage is ambiguous and prior lag experiments did not improve validation MAE.")
print("6. yearID remains useful for EDA and year-aware validation.")
print("7. Next: 01_eda.ipynb")


,decision,value
0,Data-quality status,USABLE WITH CAVEATS
1,Structural blocker checks,0
2,Warning checks,4
3,Use ID as row identifier,True
4,Use numeric season statistics for EDA/modeling,True
5,Use W as supervised target,True
6,Use teamID/franchID as model predictors,False
7,Use yearID for EDA and year-aware validation,True
8,Use franchise/team lag features,False


,severity,check,count,message
0,WARNING,ambiguous_franchise_year,384,franchID + yearID is not unique; franchise-bas...
1,WARNING,ambiguous_franchise_year,384,franchID + yearID is not unique; franchise-bas...
2,WARNING,cross_file_team_year_overlap,361,teamID + yearID overlaps across train/predicti...
3,WARNING,cross_file_team_year_overlap,361,teamID + yearID overlaps across train/predicti...



Interpretation:
1. ID is the unique row-level identifier.
2. teamID, franchID and yearID are descriptive fields, not global row keys.
3. teamID + yearID overlaps across train and prediction, so that pair is not a global observation key.
4. Some franchID + yearID combinations are not unique.
5. Lag features are excluded because historical linkage is ambiguous and prior lag experiments did not improve validation MAE.
6. yearID remains useful for EDA and year-aware validation.
7. Next: 01_eda.ipynb


### Decisions

The supplied datasets are structurally usable for EDA and modelling.

`ID` is the row-level identifier: it is unique within each file and no `ID` appears in both train and prediction.

`teamID`, `franchID`, and `yearID` are useful descriptive fields, but their combinations are not reliable as global row keys across the two files. In particular, some team/year combinations appear in both train and prediction as different observations, and some franchise/year combinations are not unique.

This matters for lag features. A lag feature uses values from a previous season for the same team or franchise. Because the franchise/year mapping is not consistently unique, constructing those lags would require extra reconciliation and could create ambiguous links between seasons.

For the main modelling pipeline, lag features are therefore excluded. This is also consistent with the earlier experiments, where lag features did not improve validation MAE.

## 6. Save diagnostic reports

Reports are written to `outputs/data_quality/`. Raw datasets are never overwritten.


In [14]:
schema_summary.to_csv(REPORT_DIR / "schema_summary.csv", index=False)
missingness.to_csv(REPORT_DIR / "missingness.csv", index=False)
key_summary.to_csv(REPORT_DIR / "key_summary.csv", index=False)
chronology_summary.to_csv(REPORT_DIR / "chronology_summary.csv", index=False)
issues_df.to_csv(REPORT_DIR / "data_quality_issues.csv", index=False)
decision.to_csv(REPORT_DIR / "data_quality_decision.csv", index=False)

if not overlap_comparison.empty:
    overlap_comparison.to_csv(
        REPORT_DIR / "team_year_overlap_comparison.csv",
        index=False,
    )

summary = {
    "status": status,
    "train_rows": int(len(train)),
    "prediction_rows": int(len(pred)),
    "structural_blocker_checks": blocker_count,
    "warning_checks": warning_count,
    "identity_constrained": identity_constrained,
    "safe_for_eda_and_modeling": blocker_count == 0,
    "safe_for_franchise_lags": blocker_count == 0 and not identity_constrained,
}

(REPORT_DIR / "data_quality_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(f"Reports written to: {REPORT_DIR}")
print(json.dumps(summary, indent=2))


Reports written to: /home/shpang/devs/ntu/projects/baseball_v2/outputs/data_quality
{
  "status": "USABLE WITH CAVEATS",
  "train_rows": 1812,
  "prediction_rows": 453,
  "structural_blocker_checks": 0,
  "warning_checks": 4,
  "identity_constrained": true,
  "safe_for_eda_and_modeling": true,
  "safe_for_franchise_lags": false
}
